# 14 — Overfitting, robustness, and leakage audit

This rerun diagnoses the LightGBM baseline and a small controlled set of regularized LightGBM candidates using development data only. It does not optimize thresholds, calibrate probabilities, alter the split, or evaluate the test set.

### What this cell does
Maps the required development artifacts, defines diagnostic report locations, and fingerprints all protected inputs and the baseline LightGBM.

### Why it matters
A diagnostic audit must operate on the established pipeline and prove that preprocessing, the baseline model, and development data are unchanged.

### What to understand
Only official train and validation content is loaded; test files are neither read nor evaluated.

In [1]:
from pathlib import Path
import hashlib
import json
import os
import re
import time

import joblib
import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from lightgbm import LGBMClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
assert ROOT.name == "AdoptAI_V1"
PREPROCESSED_DIR = ROOT / "data/modeling/preprocessed"
REPORT_DIR = ROOT / "reports"
FIGURE_DIR = REPORT_DIR / "figures/robustness"
DIAGNOSTIC_MODEL_DIR = ROOT / "models/diagnostic"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
DIAGNOSTIC_MODEL_DIR.mkdir(parents=True, exist_ok=True)

paths = {
    "feature_dataset": ROOT / "data/interim/feature_dataset.csv",
    "train_split": ROOT / "data/modeling/train.csv",
    "validation_split": ROOT / "data/modeling/validation.csv",
    "X_train": PREPROCESSED_DIR / "X_train_tree.csv",
    "X_validation": PREPROCESSED_DIR / "X_validation_tree.csv",
    "y_train": PREPROCESSED_DIR / "y_train.csv",
    "y_validation": PREPROCESSED_DIR / "y_validation.csv",
    "train_identifiers": PREPROCESSED_DIR / "train_identifiers.csv",
    "validation_identifiers": PREPROCESSED_DIR / "validation_identifiers.csv",
    "baseline_lightgbm": ROOT / "models/baseline/lightgbm.joblib",
    "tree_preprocessor": ROOT / "models/preprocessing/tree_preprocessor.joblib",
    "feature_notebook": ROOT / "notebooks/08_feature_engineering.ipynb",
    "preprocessing_notebook": ROOT / "notebooks/11_preprocessing.ipynb",
    "final_split_summary": REPORT_DIR / "final_split_summary.csv",
    "final_split_by_run": REPORT_DIR / "final_split_by_run.csv",
}
assert all(path.is_file() for path in paths.values())

def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

protected_hashes_before = {name: sha256_file(path) for name, path in paths.items()}
print(f"Existing artifacts mapped: {len(paths)}; protected fingerprints recorded.")
print("Validation-only boundary established; no test artifact is referenced or opened.")

Existing artifacts mapped: 15; protected fingerprints recorded.
Validation-only boundary established; no test artifact is referenced or opened.


### What this cell does
Loads train and validation feature matrices, targets, identifiers, split metadata, and the new baseline LightGBM, then validates row alignment and class distributions.

### Why it matters
Every robustness result depends on exact alignment between model rows, labels, machines, runs, and timestamps.

### What to understand
Passing checks reproduce the new 82,860-row train and 22,212-row validation datasets without refitting preprocessing.

In [2]:
X_train = pd.read_csv(paths["X_train"])
X_validation = pd.read_csv(paths["X_validation"])
y_train = pd.read_csv(paths["y_train"])["slowdown_in_5min"].astype(int)
y_validation = pd.read_csv(paths["y_validation"])["slowdown_in_5min"].astype(int)
train_identifiers = pd.read_csv(paths["train_identifiers"])
validation_identifiers = pd.read_csv(paths["validation_identifiers"])
split_summary = pd.read_csv(paths["final_split_summary"]).set_index("split")
split_by_run = pd.read_csv(paths["final_split_by_run"])
baseline_lightgbm = joblib.load(paths["baseline_lightgbm"])

assert X_train.shape == (82_860, 360) and X_validation.shape == (22_212, 360)
assert X_train.columns.tolist() == X_validation.columns.tolist()
assert len(X_train) == len(y_train) == len(train_identifiers)
assert len(X_validation) == len(y_validation) == len(validation_identifiers)
assert not X_train.isna().any().any() and not X_validation.isna().any().any()
assert np.isfinite(X_train.to_numpy(dtype=float)).all() and np.isfinite(X_validation.to_numpy(dtype=float)).all()
assert int(y_train.sum()) == int(split_summary.loc["train", "positive_rows"])
assert int(y_validation.sum()) == int(split_summary.loc["validation", "positive_rows"])
test_run_ids_from_assignment_report = set(split_by_run.loc[split_by_run["split"].eq("test"), "run_id"])
development_runs = set(train_identifiers["run_id"]) | set(validation_identifiers["run_id"])
assert test_run_ids_from_assignment_report.isdisjoint(development_runs)
print(f"Train: {X_train.shape}, positive rate={y_train.mean():.2%}")
print(f"Validation: {X_validation.shape}, positive rate={y_validation.mean():.2%}")
print("No test features, labels, predictions, or test-derived metrics were loaded.")

Train: (82860, 360), positive rate=34.03%
Validation: (22212, 360), positive rate=49.13%
No test features, labels, predictions, or test-derived metrics were loaded.


### What this cell does
Audits transformed feature names, exact target proxies, and the feature-engineering source code for future-looking tokens, centered windows, negative shifts, and boundary violations.

### Why it matters
High validation performance is unacceptable if any current-row feature contains target or future information, even when its name looks harmless.

### What to understand
The audit combines name checks, value-level proxy checks, and implementation inspection; any confirmed leakage stops the notebook before model diagnostics.

In [3]:
feature_names = X_train.columns.tolist()
suspicious_tokens = ["future", "target", "label", "slowdown", "horizon", "valid", "rule_c"]
suspicious_names = [name for name in feature_names if any(token in name.lower() for token in suspicious_tokens)]
forbidden_exact = {"slowdown_in_5min", "slowdown_in_10min", "slowdown_now", "valid_5min_horizon",
                   "valid_10min_horizon", "machine_id", "run_id", "segment_id", "timestamp", "id"}
forbidden_present = sorted(set(feature_names) & forbidden_exact)
rule_c_features = [name for name in feature_names if name.startswith(("moderate_", "severe_", "rule_a_", "rule_b_", "rule_c_"))]
exact_target_proxies = []
target_values = y_train.to_numpy(dtype=float)
for name in feature_names:
    values = X_train[name].to_numpy(dtype=float)
    if np.array_equal(values, target_values) or np.array_equal(values, 1 - target_values):
        exact_target_proxies.append(name)

feature_notebook = json.loads(paths["feature_notebook"].read_text())
feature_source = "\n".join("".join(cell.get("source", [])) for cell in feature_notebook["cells"] if cell.get("cell_type") == "code")
source_checks = {
    "rolling_windows_not_centered": not bool(re.search(r"center\s*=\s*True", feature_source)),
    "no_negative_shift": not bool(re.search(r"shift\s*\(\s*-", feature_source)),
    "explicit_sequence_boundaries": 'sequence_keys = ["machine_id", "run_id", "segment_id"]' in feature_source,
    "rolling_grouped_by_sequence": "for _, segment in feature_work.groupby(sequence_keys" in feature_source,
    "differences_grouped_by_sequence": "groupby(sequence_keys, sort=False)[metric].diff()" in feature_source,
    "past_window_right_closed": 'rolling(window, min_periods=1, closed="right")' in feature_source,
    "sampled_past_window_assertion": '_timestamp_dt"].gt(start_time)' in feature_source and '_timestamp_dt"].le(row["_timestamp_dt"])' in feature_source,
}
audit_rows = [
    {"feature_or_rule": "forbidden target/identifier columns", "status": "pass" if not forbidden_present else "fail", "reason": f"Found: {forbidden_present}", "risk_level": "critical" if forbidden_present else "none"},
    {"feature_or_rule": "suspicious future/target tokens", "status": "pass" if not suspicious_names else "review", "reason": f"Found: {suspicious_names}", "risk_level": "high" if suspicious_names else "none"},
    {"feature_or_rule": "Rule C intermediate features", "status": "pass" if not rule_c_features else "fail", "reason": f"Found: {rule_c_features}", "risk_level": "critical" if rule_c_features else "none"},
    {"feature_or_rule": "exact target or inverse-target value proxy", "status": "pass" if not exact_target_proxies else "fail", "reason": f"Found: {exact_target_proxies}", "risk_level": "critical" if exact_target_proxies else "none"},
]
for rule, passed in source_checks.items():
    audit_rows.append({"feature_or_rule": rule, "status": "pass" if passed else "fail",
                       "reason": "Implementation satisfies the causal check." if passed else "Required causal pattern was not verified.",
                       "risk_level": "none" if passed else "critical"})
leakage_audit = pd.DataFrame(audit_rows)
leakage_audit_path = REPORT_DIR / "feature_leakage_audit.csv"
leakage_audit.to_csv(leakage_audit_path, index=False)
display(leakage_audit)
real_leakage_found = leakage_audit["status"].eq("fail").any()
if real_leakage_found:
    raise RuntimeError("Confirmed or unresolved critical feature leakage detected; robustness analysis stopped.")
print(f"Real leakage found: {real_leakage_found}")

,feature_or_rule,status,reason,risk_level
0,forbidden target/identifier columns,pass,Found: [],none
1,suspicious future/target tokens,pass,Found: [],none
2,Rule C intermediate features,pass,Found: [],none
3,exact target or inverse-target value proxy,pass,Found: [],none
4,rolling_windows_not_centered,pass,Implementation satisfies the causal check.,none
5,no_negative_shift,pass,Implementation satisfies the causal check.,none
6,explicit_sequence_boundaries,pass,Implementation satisfies the causal check.,none
7,rolling_grouped_by_sequence,pass,Implementation satisfies the causal check.,none
8,differences_grouped_by_sequence,pass,Implementation satisfies the causal check.,none
9,past_window_right_closed,pass,Implementation satisfies the causal check.,none


Real leakage found: False


### What this cell does
Evaluates the new baseline LightGBM on train and validation data at threshold 0.50 and records metric gaps.

### Why it matters
Overfitting interpretation should use the size and consistency of multiple metric gaps rather than treating a train PR-AUC near one as decisive by itself.

### What to understand
Positive gaps mean train performance is higher; validation remains the relevant development estimate despite its shifted class distribution.

In [4]:
THRESHOLD = 0.50
RANDOM_SEED = 42
N_JOBS = min(4, max(1, (os.cpu_count() or 2) // 2))

def calculate_metrics(y_true, probability):
    prediction = (probability >= THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {"pr_auc": average_precision_score(y_true, probability),
            "roc_auc": roc_auc_score(y_true, probability) if pd.Series(y_true).nunique() == 2 else np.nan,
            "accuracy": accuracy_score(y_true, prediction),
            "precision": precision_score(y_true, prediction, zero_division=0),
            "recall": recall_score(y_true, prediction, zero_division=0),
            "f1": f1_score(y_true, prediction, zero_division=0),
            "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)}

official_models = {"LightGBM": baseline_lightgbm}
official_probabilities, metric_gap_rows = {}, []
for model_name, model in official_models.items():
    train_probability = model.predict_proba(X_train)[:, 1]
    validation_probability = model.predict_proba(X_validation)[:, 1]
    official_probabilities[model_name] = {"train": train_probability, "validation": validation_probability}
    train_metrics = calculate_metrics(y_train, train_probability)
    validation_metrics = calculate_metrics(y_validation, validation_probability)
    for metric in ["pr_auc", "roc_auc", "precision", "recall", "f1", "tn", "fp", "fn", "tp"]:
        metric_gap_rows.append({"model": model_name, "metric": metric, "train_metric": train_metrics[metric],
                                "validation_metric": validation_metrics[metric], "gap_train_minus_validation": train_metrics[metric] - validation_metrics[metric]})
train_validation_gaps = pd.DataFrame(metric_gap_rows)
train_validation_gaps_path = REPORT_DIR / "train_validation_metric_gaps.csv"
train_validation_gaps.to_csv(train_validation_gaps_path, index=False)
display(train_validation_gaps)

,model,metric,train_metric,validation_metric,gap_train_minus_validation
0,LightGBM,pr_auc,0.999996,0.968988,0.031008
1,LightGBM,roc_auc,0.999998,0.958868,0.041130
2,LightGBM,precision,0.997806,0.895394,0.102411
3,LightGBM,recall,0.999858,0.899661,0.100197
4,LightGBM,f1,0.998831,0.897523,0.101308
5,LightGBM,tn,54600.000000,10152.000000,44448.000000
6,LightGBM,fp,62.000000,1147.000000,-1085.000000
7,LightGBM,fn,4.000000,1095.000000,-1091.000000
8,LightGBM,tp,28194.000000,9818.000000,18376.000000


### What this cell does
Calculates validation performance separately by machine and by complete run, including explicit weakness, size, class-coverage, recall, and dominance flags.

### Why it matters
A strong global score can be dominated by one large or easy run and can hide machines where the model is unreliable.

### What to understand
Subgroup PR-AUC and recall are compared with each model's global validation values; single-class groups are flagged rather than silently treated as ordinary.

In [5]:
validation_context = validation_identifiers[["machine_id", "run_id", "segment_id", "timestamp"]].copy()
validation_context["true_target"] = y_validation.to_numpy()
for model_name in official_models:
    validation_context[f"{model_name}_probability"] = official_probabilities[model_name]["validation"]
global_validation_metrics = {name: calculate_metrics(y_validation, values["validation"]) for name, values in official_probabilities.items()}

def subgroup_records(group_columns, level):
    records = []
    grouping = group_columns[0] if len(group_columns) == 1 else group_columns
    for keys, group in validation_context.groupby(grouping, sort=False):
        keys = (keys,) if len(group_columns) == 1 else keys
        positives = int(group["true_target"].sum()); negatives = int(len(group) - positives)
        for model_name in official_models:
            metrics = calculate_metrics(group["true_target"], group[f"{model_name}_probability"])
            record = {column: value for column, value in zip(group_columns, keys)}
            record.update({"level": level, "model": model_name, "rows": len(group),
                           "positive_count": positives, "negative_count": negatives,
                           "positive_rate": positives / len(group), **metrics})
            record["extremely_small_group"] = len(group) < 500
            record["single_class_group"] = positives == 0 or negatives == 0
            record["substantially_weaker_pr_auc"] = metrics["pr_auc"] < global_validation_metrics[model_name]["pr_auc"] - 0.05
            record["unusually_poor_recall"] = metrics["recall"] < global_validation_metrics[model_name]["recall"] - 0.15
            record["dominates_validation_rows"] = len(group) / len(validation_context) >= 0.40
            record["dominates_validation_positives"] = positives / max(1, int(y_validation.sum())) >= 0.40
            records.append(record)
    return records

robustness_by_machine = pd.DataFrame(subgroup_records(["machine_id"], "machine"))
robustness_by_run = pd.DataFrame(subgroup_records(["machine_id", "run_id"], "run"))
machine_path = REPORT_DIR / "robustness_by_machine.csv"; run_path = REPORT_DIR / "robustness_by_run.csv"
robustness_by_machine.to_csv(machine_path, index=False); robustness_by_run.to_csv(run_path, index=False)
print("By machine:"); display(robustness_by_machine)
print("By run:"); display(robustness_by_run)

By machine:


/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


,machine_id,level,model,rows,positive_count,negative_count,positive_rate,pr_auc,roc_auc,accuracy,...,tn,fp,fn,tp,extremely_small_group,single_class_group,substantially_weaker_pr_auc,unusually_poor_recall,dominates_validation_rows,dominates_validation_positives
0,0890dcc046c079acc4de4202,machine,LightGBM,3024,0,3024,0.000000,0.000000,NaN,0.975529,...,2950,74,0,0,False,True,True,True,False,False
1,7232bc533c21ce408d45d473,machine,LightGBM,2304,884,1420,0.383681,0.836177,0.845611,0.741319,...,969,451,145,739,False,False,True,False,False,False
2,a0f8c86097e55fbfa506d057,machine,LightGBM,8607,1752,6855,0.203555,0.574466,0.746262,0.817707,...,6233,622,947,805,False,False,True,True,False,False
3,d588df123ac0d0ce20b112ac,machine,LightGBM,8277,8277,0,1.000000,1.000000,NaN,0.999638,...,0,0,3,8274,False,True,False,False,False,True


By run:


,machine_id,run_id,level,model,rows,positive_count,negative_count,positive_rate,pr_auc,roc_auc,...,tn,fp,fn,tp,extremely_small_group,single_class_group,substantially_weaker_pr_auc,unusually_poor_recall,dominates_validation_rows,dominates_validation_positives
0,0890dcc046c079acc4de4202,90c048bb-46c5-4345-a113-d3bc37b876cb,run,LightGBM,3024,0,3024,0.000000,0.000000,NaN,...,2950,74,0,0,False,True,True,True,False,False
1,7232bc533c21ce408d45d473,f3c83d3f-0563-4e35-aa47-2e79402b0105,run,LightGBM,1789,472,1317,0.263835,0.750280,0.817501,...,904,413,95,377,False,False,True,False,False,False
2,7232bc533c21ce408d45d473,96f661cc-1233-4539-a814-c4c352953500,run,LightGBM,247,144,103,0.582996,0.962125,0.944647,...,65,38,1,143,True,False,False,False,False,False
3,7232bc533c21ce408d45d473,b1dd3935-56dd-4497-a77e-67128988e2c9,run,LightGBM,268,268,0,1.000000,1.000000,NaN,...,0,0,49,219,True,True,False,False,False,False
4,a0f8c86097e55fbfa506d057,0fc9db54-83d3-456a-b9ea-1aa8a6f66cf9,run,LightGBM,5,5,0,1.000000,1.000000,NaN,...,0,0,0,5,True,True,False,False,False,False
5,a0f8c86097e55fbfa506d057,baa2a6f4-7121-4bce-9c80-611ebc22ceaf,run,LightGBM,8602,1747,6855,0.203092,0.572310,0.745536,...,6233,622,947,800,False,False,True,True,False,False
6,d588df123ac0d0ce20b112ac,35472c29-8dd3-49be-8d6b-1984d158745d,run,LightGBM,8277,8277,0,1.000000,1.000000,NaN,...,0,0,3,8274,False,True,False,False,False,True


### What this cell does
Fits six controlled regularized LightGBM candidates—including the previously successful settings—and one separate early-stopping diagnostic, always using train for fitting and validation for monitoring.

### Why it matters
These experiments test whether lower complexity can preserve validation ranking while reducing the train–validation gap; they do not replace the official model.

### What to understand
Validation PR-AUC remains primary. Lower training performance is useful only when validation performance is retained or improved.

In [6]:
train_ratio = float((y_train == 0).sum() / (y_train == 1).sum())
lgbm_candidates = [
    {"max_depth": 7, "num_leaves": 15, "min_child_samples": 100, "reg_alpha": 0.1, "reg_lambda": 0.1, "subsample": 0.9, "colsample_bytree": 0.8},
    {"max_depth": 5, "num_leaves": 15, "min_child_samples": 100, "reg_alpha": 0.1, "reg_lambda": 0.1, "subsample": 0.9, "colsample_bytree": 0.8},
    {"max_depth": 7, "num_leaves": 15, "min_child_samples": 150, "reg_alpha": 0.5, "reg_lambda": 0.5, "subsample": 0.9, "colsample_bytree": 0.8},
    {"max_depth": 7, "num_leaves": 31, "min_child_samples": 100, "reg_alpha": 0.5, "reg_lambda": 0.5, "subsample": 0.8, "colsample_bytree": 0.8},
    {"max_depth": 10, "num_leaves": 31, "min_child_samples": 100, "reg_alpha": 1.0, "reg_lambda": 1.0, "subsample": 0.9, "colsample_bytree": 0.9},
    {"max_depth": 7, "num_leaves": 15, "min_child_samples": 200, "reg_alpha": 1.0, "reg_lambda": 1.0, "subsample": 0.8, "colsample_bytree": 0.8},
]
regularized_lgbm_rows = []
for trial_number, parameters in enumerate(lgbm_candidates, start=1):
    started = time.perf_counter()
    model = LGBMClassifier(objective="binary", n_estimators=300, learning_rate=0.05,
                           scale_pos_weight=train_ratio, subsample_freq=1, random_state=RANDOM_SEED,
                           n_jobs=N_JOBS, verbosity=-1, **parameters)
    model.fit(X_train, y_train)
    train_metrics = calculate_metrics(y_train, model.predict_proba(X_train)[:, 1])
    validation_metrics = calculate_metrics(y_validation, model.predict_proba(X_validation)[:, 1])
    regularized_lgbm_rows.append({"experiment": "regularized_random_search", "candidate": trial_number,
                                  "parameters_json": json.dumps(parameters, sort_keys=True),
                                  "train_pr_auc": train_metrics["pr_auc"], "validation_pr_auc": validation_metrics["pr_auc"],
                                  "train_validation_pr_auc_gap": train_metrics["pr_auc"] - validation_metrics["pr_auc"],
                                  "validation_roc_auc": validation_metrics["roc_auc"], "precision": validation_metrics["precision"],
                                  "recall": validation_metrics["recall"], "f1": validation_metrics["f1"],
                                  "fp": validation_metrics["fp"], "fn": validation_metrics["fn"],
                                  "best_iteration": np.nan, "execution_time_seconds": time.perf_counter() - started})
    print(f"Regularized LightGBM {trial_number:02d}/{len(lgbm_candidates)}: val PR-AUC={validation_metrics['pr_auc']:.6f}")

early_started = time.perf_counter()
early_model = LGBMClassifier(objective="binary", n_estimators=1000, learning_rate=0.05, num_leaves=31,
                             max_depth=-1, min_child_samples=20, scale_pos_weight=train_ratio,
                             random_state=RANDOM_SEED, n_jobs=N_JOBS, verbosity=-1)
early_model.fit(X_train, y_train, eval_set=[(X_validation, y_validation)], eval_metric="average_precision",
                callbacks=[lgb.early_stopping(50, first_metric_only=True, verbose=False), lgb.log_evaluation(0)])
early_train_metrics = calculate_metrics(y_train, early_model.predict_proba(X_train)[:, 1])
early_validation_metrics = calculate_metrics(y_validation, early_model.predict_proba(X_validation)[:, 1])
regularized_lgbm_rows.append({"experiment": "early_stopping", "candidate": 1,
                              "parameters_json": json.dumps({"n_estimators": 1000, "learning_rate": 0.05, "early_stopping_rounds": 50}, sort_keys=True),
                              "train_pr_auc": early_train_metrics["pr_auc"], "validation_pr_auc": early_validation_metrics["pr_auc"],
                              "train_validation_pr_auc_gap": early_train_metrics["pr_auc"] - early_validation_metrics["pr_auc"],
                              "validation_roc_auc": early_validation_metrics["roc_auc"], "precision": early_validation_metrics["precision"],
                              "recall": early_validation_metrics["recall"], "f1": early_validation_metrics["f1"],
                              "fp": early_validation_metrics["fp"], "fn": early_validation_metrics["fn"],
                              "best_iteration": int(early_model.best_iteration_), "execution_time_seconds": time.perf_counter() - early_started})
regularized_lightgbm_results = pd.DataFrame(regularized_lgbm_rows).sort_values("validation_pr_auc", ascending=False).reset_index(drop=True)
regularized_lightgbm_path = REPORT_DIR / "regularized_lightgbm_results.csv"
regularized_lightgbm_results.to_csv(regularized_lightgbm_path, index=False)
display(regularized_lightgbm_results)

Regularized LightGBM 01/6: val PR-AUC=0.964754


Regularized LightGBM 02/6: val PR-AUC=0.965079


Regularized LightGBM 03/6: val PR-AUC=0.964946


Regularized LightGBM 04/6: val PR-AUC=0.966662


Regularized LightGBM 05/6: val PR-AUC=0.967594


Regularized LightGBM 06/6: val PR-AUC=0.965822


,experiment,candidate,parameters_json,train_pr_auc,validation_pr_auc,train_validation_pr_auc_gap,validation_roc_auc,precision,recall,f1,fp,fn,best_iteration,execution_time_seconds
0,early_stopping,1,"{""early_stopping_rounds"": 50, ""learning_rate"":...",0.992359,0.970804,0.021555,0.963428,0.851673,0.904976,0.877516,1720,1037,60.0,2.268454
1,regularized_random_search,5,"{""colsample_bytree"": 0.9, ""max_depth"": 10, ""mi...",0.999960,0.967594,0.032366,0.956799,0.884942,0.897187,0.891022,1273,1122,NaN,4.320111
2,regularized_random_search,4,"{""colsample_bytree"": 0.8, ""max_depth"": 7, ""min...",0.999898,0.966662,0.033236,0.954735,0.888909,0.893796,0.891346,1219,1159,NaN,3.491007
3,regularized_random_search,6,"{""colsample_bytree"": 0.8, ""max_depth"": 7, ""min...",0.998551,0.965822,0.032729,0.951991,0.869910,0.904426,0.886832,1476,1043,NaN,3.203927
4,regularized_random_search,2,"{""colsample_bytree"": 0.8, ""max_depth"": 5, ""min...",0.997905,0.965079,0.032826,0.951604,0.878508,0.897828,0.888063,1355,1115,NaN,2.570180
5,regularized_random_search,3,"{""colsample_bytree"": 0.8, ""max_depth"": 7, ""min...",0.998806,0.964946,0.033861,0.951133,0.835504,0.905251,0.868980,1945,1034,NaN,3.340666
6,regularized_random_search,1,"{""colsample_bytree"": 0.8, ""max_depth"": 7, ""min...",0.998880,0.964754,0.034126,0.950944,0.860459,0.900119,0.879842,1593,1090,NaN,3.148237


### What this cell does
Selects the strongest controlled regularized LightGBM candidate, rebuilds it with the complete parameter set, and saves it as a diagnostic model.

### Why it matters
The saved model must correspond exactly to the candidate evaluated in later robustness and threshold notebooks.

### What to understand
This remains a development artifact; validation performance does not make it a final production model.

In [7]:
best_regularized_row = (regularized_lightgbm_results.loc[regularized_lightgbm_results["experiment"].eq("regularized_random_search")]
                        .sort_values(["validation_pr_auc", "train_validation_pr_auc_gap"], ascending=[False, True]).iloc[0])
best_regularized_parameters = json.loads(best_regularized_row["parameters_json"])
complete_regularized_parameters = {"objective": "binary", "n_estimators": 300, "learning_rate": 0.05,
                                   "scale_pos_weight": train_ratio, "subsample_freq": 1,
                                   "random_state": RANDOM_SEED, "n_jobs": N_JOBS, "verbosity": -1,
                                   **best_regularized_parameters}
best_regularized_model = LGBMClassifier(**complete_regularized_parameters)
best_regularized_model.fit(X_train, y_train)
diagnostic_model_path = DIAGNOSTIC_MODEL_DIR / "regularized_lightgbm.joblib"
diagnostic_parameters_path = DIAGNOSTIC_MODEL_DIR / "regularized_lightgbm_parameters.json"
joblib.dump(best_regularized_model, diagnostic_model_path)
diagnostic_parameters_path.write_text(json.dumps(complete_regularized_parameters, indent=2, sort_keys=True) + "\n")
print(f"Selected controlled candidate {int(best_regularized_row['candidate'])}: validation PR-AUC={best_regularized_row['validation_pr_auc']:.6f}")
print(f"Saved {diagnostic_model_path} and {diagnostic_parameters_path}")

Selected controlled candidate 5: validation PR-AUC=0.967594
Saved /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/diagnostic/regularized_lightgbm.joblib and /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/models/diagnostic/regularized_lightgbm_parameters.json


### What this cell does
Creates one temporary scenario for each of the seven complete validation runs using the unchanged training set and baseline LightGBM configuration.

### Why it matters
This reveals whether the global validation result persists when the dominant validation runs and machines change, without using any official test run.

### What to understand
These are diagnostic scenarios, not replacement splits. Each scenario contains one entire validation run and preserves the official train-to-validation boundary.

In [8]:
scenario_rows = []
validation_run_ids = validation_identifiers["run_id"].drop_duplicates().tolist()
assert len(validation_run_ids) == 7 and set(validation_run_ids).isdisjoint(test_run_ids_from_assignment_report)
scenario_parameters = {"n_estimators": 300, "learning_rate": 0.05, "num_leaves": 31,
                       "max_depth": -1, "min_child_samples": 20, "scale_pos_weight": train_ratio}
for scenario_number, run_id in enumerate(validation_run_ids, start=1):
    mask = validation_identifiers["run_id"].eq(run_id).to_numpy()
    assert mask.any() and set(validation_identifiers.loc[mask, "run_id"]) == {run_id}
    model = LGBMClassifier(objective="binary", random_state=RANDOM_SEED, n_jobs=N_JOBS, verbosity=-1, **scenario_parameters)
    model.fit(X_train, y_train)
    probability = model.predict_proba(X_validation.loc[mask])[:, 1]
    metrics = calculate_metrics(y_validation.loc[mask], probability)
    machines = sorted(validation_identifiers.loc[mask, "machine_id"].unique().tolist())
    scenario_rows.append({"scenario": f"validation_run_{scenario_number}", "validation_run_ids": json.dumps([run_id]),
                          "training_rows": len(X_train), "validation_rows": int(mask.sum()),
                          "validation_positive_rate": float(y_validation.loc[mask].mean()),
                          "machines_represented": json.dumps(machines), "machine_count": len(machines),
                          "pr_auc": metrics["pr_auc"], "roc_auc": metrics["roc_auc"],
                          "precision": metrics["precision"], "recall": metrics["recall"], "f1": metrics["f1"],
                          "fp": metrics["fp"], "fn": metrics["fn"]})
temporal_robustness_scenarios = pd.DataFrame(scenario_rows)
temporal_path = REPORT_DIR / "temporal_robustness_scenarios.csv"
temporal_robustness_scenarios.to_csv(temporal_path, index=False)
display(temporal_robustness_scenarios)

/Users/fatimazahranamaoui/Documents/AdoptAI_Project/adoptai_env/lib/python3.13/site-packages/sklearn/metrics/_ranking.py:1192: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(


,scenario,validation_run_ids,training_rows,validation_rows,validation_positive_rate,machines_represented,machine_count,pr_auc,roc_auc,precision,recall,f1,fp,fn
0,validation_run_1,"[""90c048bb-46c5-4345-a113-d3bc37b876cb""]",82860,3024,0.000000,"[""0890dcc046c079acc4de4202""]",1,0.000000,NaN,0.000000,0.000000,0.000000,74,0
1,validation_run_2,"[""f3c83d3f-0563-4e35-aa47-2e79402b0105""]",82860,1789,0.263835,"[""7232bc533c21ce408d45d473""]",1,0.750280,0.817501,0.477215,0.798729,0.597464,413,95
2,validation_run_3,"[""96f661cc-1233-4539-a814-c4c352953500""]",82860,247,0.582996,"[""7232bc533c21ce408d45d473""]",1,0.962125,0.944647,0.790055,0.993056,0.880000,38,1
3,validation_run_4,"[""b1dd3935-56dd-4497-a77e-67128988e2c9""]",82860,268,1.000000,"[""7232bc533c21ce408d45d473""]",1,1.000000,NaN,1.000000,0.817164,0.899384,0,49
4,validation_run_5,"[""0fc9db54-83d3-456a-b9ea-1aa8a6f66cf9""]",82860,5,1.000000,"[""a0f8c86097e55fbfa506d057""]",1,1.000000,NaN,1.000000,1.000000,1.000000,0,0
5,validation_run_6,"[""baa2a6f4-7121-4bce-9c80-611ebc22ceaf""]",82860,8602,0.203092,"[""a0f8c86097e55fbfa506d057""]",1,0.572310,0.745536,0.562588,0.457928,0.504891,622,947
6,validation_run_7,"[""35472c29-8dd3-49be-8d6b-1984d158745d""]",82860,8277,1.000000,"[""d588df123ac0d0ce20b112ac""]",1,1.000000,NaN,1.000000,0.999638,0.999819,0,3


### What this cell does
Synthesizes the leakage, subgroup, regularization, early-stopping, and run-scenario evidence into one explicit diagnostic category and saves the required summary.

### Why it matters
The conclusion must reflect stability across runs and machines and the unusual validation distribution—not merely perfect training PR-AUC.

### What to understand
Category D is used when the class-distribution shift and subgroup/scenario variability prevent a confident generalization claim despite otherwise strong metrics.

In [9]:
lightgbm_pr_gap = float(train_validation_gaps.query("model == 'LightGBM' and metric == 'pr_auc'")["gap_train_minus_validation"].iloc[0])
machine_pr_range = float(robustness_by_machine.groupby("model")["pr_auc"].agg(lambda values: values.max() - values.min()).max())
run_pr_range = float(robustness_by_run.groupby("model")["pr_auc"].agg(lambda values: values.max() - values.min()).max())
scenario_pr_range = float(temporal_robustness_scenarios["pr_auc"].max() - temporal_robustness_scenarios["pr_auc"].min())
validation_shift = abs(float(y_validation.mean() - y_train.mean()))
best_regularized_lgbm = regularized_lightgbm_results.iloc[0]
eligible_run_metrics = robustness_by_run.loc[~robustness_by_run["single_class_group"] & ~robustness_by_run["extremely_small_group"]]
minimum_run_pr_auc = float(eligible_run_metrics["pr_auc"].min())
minimum_run_recall = float(eligible_run_metrics["recall"].min())
strong_instability = bool(minimum_run_pr_auc < 0.65 or minimum_run_recall < 0.40)
subgroup_instability = bool(machine_pr_range >= 0.05 or run_pr_range >= 0.05 or scenario_pr_range >= 0.05)
if real_leakage_found:
    evidence_category = "A. Strong evidence of harmful overfitting"
    conclusion_reason = "Confirmed feature leakage invalidates the apparent model performance."
elif strong_instability:
    evidence_category = "D. Results inconclusive because validation distribution is too unusual"
    conclusion_reason = "No leakage was found and global gaps are modest, but the large prevalence shift and subgroup/run variability prevent a stable generalization conclusion."
elif lightgbm_pr_gap >= 0.03:
    evidence_category = "B. Mild overfitting but validation remains robust"
    conclusion_reason = "Train performance is higher, but validation and robustness experiments remain comparatively strong."
else:
    evidence_category = "C. No meaningful overfitting detected"
    conclusion_reason = "No leakage was found, train-validation gaps are small, and subgroup/scenario performance is sufficiently stable."
robustness_summary = pd.DataFrame([{
    "evidence_category": evidence_category, "conclusion_reason": conclusion_reason,
    "real_feature_leakage_found": real_leakage_found, "train_positive_rate": y_train.mean(),
    "validation_positive_rate": y_validation.mean(),
    "lightgbm_train_validation_pr_auc_gap": lightgbm_pr_gap,
    "maximum_machine_pr_auc_range": machine_pr_range, "maximum_run_pr_auc_range": run_pr_range,
    "temporal_scenario_pr_auc_range": scenario_pr_range,
    "weak_machine_rows": int(robustness_by_machine["substantially_weaker_pr_auc"].sum()),
    "weak_run_rows": int(robustness_by_run["substantially_weaker_pr_auc"].sum()),
    "best_regularized_lightgbm_validation_pr_auc": best_regularized_lgbm["validation_pr_auc"],
    "minimum_eligible_run_pr_auc": minimum_run_pr_auc, "minimum_eligible_run_recall": minimum_run_recall,
    "strong_instability_detected": strong_instability,
    "threshold_optimization_allowed": not real_leakage_found and not strong_instability,
    "early_stopping_best_iteration": int(regularized_lightgbm_results.loc[regularized_lightgbm_results["experiment"].eq("early_stopping"), "best_iteration"].iloc[0]),
}])
summary_path = REPORT_DIR / "overfitting_robustness_summary.csv"
robustness_summary.to_csv(summary_path, index=False)
display(robustness_summary)

,evidence_category,conclusion_reason,real_feature_leakage_found,train_positive_rate,validation_positive_rate,lightgbm_train_validation_pr_auc_gap,maximum_machine_pr_auc_range,maximum_run_pr_auc_range,temporal_scenario_pr_auc_range,weak_machine_rows,weak_run_rows,best_regularized_lightgbm_validation_pr_auc,minimum_eligible_run_pr_auc,minimum_eligible_run_recall,strong_instability_detected,threshold_optimization_allowed,early_stopping_best_iteration
0,D. Results inconclusive because validation dis...,No leakage was found and global gaps are modes...,False,0.340309,0.491311,0.031008,1.0,1.0,1.0,3,3,0.970804,0.57231,0.457928,True,False,60


### What this cell does
Creates six diagnostic figures covering train–validation gaps, subgroup stability, regularized candidates, and temporal run scenarios.

### Why it matters
Plots reveal heterogeneity that aggregate tables can hide while remaining strictly descriptive and threshold-fixed.

### What to understand
Large variation across machines, runs, or scenarios supports caution even when global validation PR-AUC is high.

In [10]:
figure_paths = []
pr_gaps = train_validation_gaps.loc[train_validation_gaps["metric"].eq("pr_auc")]
fig, ax = plt.subplots(figsize=(7, 5)); x = np.arange(len(pr_gaps)); width = 0.36
ax.bar(x-width/2, pr_gaps["train_metric"], width, label="Train"); ax.bar(x+width/2, pr_gaps["validation_metric"], width, label="Validation")
ax.set_xticks(x, pr_gaps["model"]); ax.set_ylim(0.9, 1.005); ax.set_ylabel("PR-AUC"); ax.set_title("Train versus validation PR-AUC"); ax.legend(); ax.grid(axis="y", alpha=.25)
p=FIGURE_DIR/"train_vs_validation_pr_auc.png"; fig.tight_layout(); fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
for metric, filename, title in [("pr_auc","pr_auc_by_machine.png","Validation PR-AUC by machine"),("recall","recall_by_machine.png","Validation recall by machine")]:
    pivot=robustness_by_machine.pivot(index="machine_id",columns="model",values=metric); ax=pivot.plot(kind="bar",figsize=(9,5)); ax.set_title(title); ax.set_ylim(0,1.03); ax.grid(axis="y",alpha=.25); plt.xticks(rotation=25,ha="right"); plt.tight_layout(); p=FIGURE_DIR/filename; plt.savefig(p,dpi=160); plt.close(); figure_paths.append(p)
fig, ax=plt.subplots(figsize=(10,5)); run_plot=robustness_by_run.copy(); run_plot["short_run"]=run_plot["run_id"].str[:8];
for name,g in run_plot.groupby("model"): ax.plot(g["short_run"],g["pr_auc"],marker="o",label=name)
ax.set(title="Validation PR-AUC by complete run",ylabel="PR-AUC",xlabel="Run prefix",ylim=(0,1.03)); ax.legend(); ax.grid(alpha=.25); plt.xticks(rotation=25); fig.tight_layout(); p=FIGURE_DIR/"pr_auc_by_run.png"; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
fig,ax=plt.subplots(figsize=(9,5)); ax.scatter(regularized_lightgbm_results["train_validation_pr_auc_gap"],regularized_lightgbm_results["validation_pr_auc"],label="LightGBM candidates"); ax.set(xlabel="Train − validation PR-AUC gap",ylabel="Validation PR-AUC",title="Controlled regularized LightGBM candidates"); ax.legend(); ax.grid(alpha=.25); fig.tight_layout(); p=FIGURE_DIR/"regularization_gap_comparison.png"; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
fig,ax=plt.subplots(figsize=(8,5)); ax.bar(temporal_robustness_scenarios["scenario"],temporal_robustness_scenarios["pr_auc"]); ax.set(title="Complete-run temporal robustness scenarios",ylabel="PR-AUC",ylim=(0,1.03)); ax.grid(axis="y",alpha=.25); plt.xticks(rotation=20); fig.tight_layout(); p=FIGURE_DIR/"temporal_scenario_pr_auc.png"; fig.savefig(p,dpi=160); plt.close(fig); figure_paths.append(p)
print(f"Saved {len(figure_paths)} robustness figures under {FIGURE_DIR}")

Saved 6 robustness figures under /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/figures/robustness


### What this cell does
Reloads all required reports, verifies protected checksums and output completeness, and prints the diagnostic conclusion and stop condition.

### Why it matters
The final handoff must prove that official artifacts and test data remained untouched and that every conclusion is backed by saved evidence.

### What to understand
This notebook ends with a robustness category only; it does not select a final model or authorize threshold optimization.

In [11]:
required_reports = [summary_path, machine_path, run_path, regularized_lightgbm_path, temporal_path, leakage_audit_path, train_validation_gaps_path, diagnostic_model_path, diagnostic_parameters_path]
assert all(path.exists() and path.stat().st_size > 0 for path in required_reports + figure_paths)
assert len(pd.read_csv(machine_path)) == validation_identifiers["machine_id"].nunique()
assert len(pd.read_csv(run_path)) == validation_identifiers["run_id"].nunique()
assert len(pd.read_csv(regularized_lightgbm_path)) == len(lgbm_candidates) + 1
assert len(pd.read_csv(temporal_path)) == validation_identifiers["run_id"].nunique()
protected_hashes_after = {name: sha256_file(path) for name, path in paths.items()}
protected_unchanged = protected_hashes_before == protected_hashes_after
assert protected_unchanged
print("FINAL ROBUSTNESS REPORT")
print(f"Conclusion: {evidence_category}")
print(conclusion_reason)
print(f"Confirmed leakage found: {real_leakage_found}")
print(f"Baseline LightGBM train-validation PR-AUC gap: {lightgbm_pr_gap:.6f}")
print(f"PR-AUC ranges: machine={machine_pr_range:.6f}, run={run_pr_range:.6f}, scenarios={scenario_pr_range:.6f}")
print(f"Best regularized LightGBM validation PR-AUC: {best_regularized_lgbm['validation_pr_auc']:.6f}")
print(f"Minimum eligible-run PR-AUC={minimum_run_pr_auc:.6f}; minimum eligible-run recall={minimum_run_recall:.6f}; strong instability={strong_instability}")
early_row=regularized_lightgbm_results.loc[regularized_lightgbm_results["experiment"].eq("early_stopping")].iloc[0]
print(f"Early stopping: best iteration={int(early_row.best_iteration)}, validation PR-AUC={early_row.validation_pr_auc:.6f}, precision={early_row.precision:.4f}, recall={early_row.recall:.4f}, FP={int(early_row.fp)}, FN={int(early_row.fn)}")
print(f"WARNING: validation positive rate is {y_validation.mean():.2%}, versus {y_train.mean():.2%} in train.")
print(f"Protected official artifacts unchanged: {protected_unchanged}")
print("STOP: no threshold optimization, calibration, test evaluation, final model selection, SHAP, or dashboard work was performed.")

FINAL ROBUSTNESS REPORT
Conclusion: D. Results inconclusive because validation distribution is too unusual
No leakage was found and global gaps are modest, but the large prevalence shift and subgroup/run variability prevent a stable generalization conclusion.
Confirmed leakage found: False
Baseline LightGBM train-validation PR-AUC gap: 0.031008
PR-AUC ranges: machine=1.000000, run=1.000000, scenarios=1.000000
Best regularized LightGBM validation PR-AUC: 0.970804
Minimum eligible-run PR-AUC=0.572310; minimum eligible-run recall=0.457928; strong instability=True
Early stopping: best iteration=60, validation PR-AUC=0.970804, precision=0.8517, recall=0.9050, FP=1720, FN=1037
Protected official artifacts unchanged: True
STOP: no threshold optimization, calibration, test evaluation, final model selection, SHAP, or dashboard work was performed.
